# Lab 01 — Model Anatomy: Tokens, Logits, and Generation

**Goal:** understand the object we are about to train.

Wordle is unusually useful because it exposes a mismatch: humans naturally reason over letters, while an LLM reasons over tokens. We want to *measure* that mismatch before trying to repair it.

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tiny_wordle.hardware import preferred_device

MODEL_ID = "Qwen/Qwen3-0.6B"
device = preferred_device()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32).to(device)
model.eval()

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

## 1.1 What is a word to the tokenizer?

Tokenize several Wordle words. Do not assume one English word equals one token.

In [3]:
words = ["CRANE", "SLATE", "AUDIO", "QUEUE", "FUZZY", "jazzy", "crane"]

for word in words:
    ids = tokenizer.encode(word, add_special_tokens=False)
    pieces = [tokenizer.decode([i]) for i in ids]
    print(f"{word:8s} -> ids={ids} pieces={pieces}")

CRANE    -> ids=[8973, 27819] pieces=['CR', 'ANE']
SLATE    -> ids=[7984, 2336] pieces=['SL', 'ATE']
AUDIO    -> ids=[95666] pieces=['AUDIO']
QUEUE    -> ids=[28945] pieces=['QUEUE']
FUZZY    -> ids=[81213, 33536, 56] pieces=['FU', 'ZZ', 'Y']
jazzy    -> ids=[73, 1370, 4246] pieces=['j', 'az', 'zy']
crane    -> ids=[5082, 2145] pieces=['cr', 'ane']


Now force an explicit character representation.

In [4]:
representations = [
    "CRANE",
    "C R A N E",
    "[C][R][A][N][E]",
    "C|R|A|N|E",
]

for text in representations:
    ids = tokenizer.encode(text, add_special_tokens=False)
    pieces = [tokenizer.decode([i]) for i in ids]
    print(f"{text:20s} -> {len(ids):2d} tokens -> {pieces}")

CRANE                ->  2 tokens -> ['CR', 'ANE']
C R A N E            ->  5 tokens -> ['C', ' R', ' A', ' N', ' E']
[C][R][A][N][E]      -> 10 tokens -> ['[C', '][', 'R', '][', 'A', '][', 'N', '][', 'E', ']']
C|R|A|N|E            ->  8 tokens -> ['C', '|R', '|', 'A', '|', 'N', '|', 'E']


### Observation

Record which representation most closely exposes individual letters to the model.

Important: more tokens is not automatically better. We are identifying a representational variable we may test later.

## 1.2 Token probabilities are the primitive

Ask a simple next-token question and inspect the probability distribution directly.

In [5]:
prompt = "The five-letter English word C R A N"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    logits = model(**inputs).logits[0, -1]

probs = torch.softmax(logits.float(), dim=-1)
values, ids = torch.topk(probs, 15)

for p, token_id in zip(values.tolist(), ids.tolist()):
    print(f"{p:8.5f} {tokenizer.decode([token_id])!r}")

 0.35684 ' T'
 0.30343 ' D'
 0.13400 ' E'
 0.04384 ' G'
 0.02894 ' K'
 0.02762 ' A'
 0.02654 ' C'
 0.01281 ' S'
 0.01268 ' I'
 0.01234 ' O'
 0.00663 ' N'
 0.00638 ' H'
 0.00625 ' L'
 0.00268 ' M'
 0.00226 ' P'


## 1.3 Greedy vs sampling

Generation is a policy over those token probabilities. Compare deterministic greedy decoding with sampling.

In [10]:
def render_prompt(user_text: str, thinking: bool = False) -> str:
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": user_text}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=thinking,
    )

def generate(user_text: str, *, thinking=False, do_sample=False, temperature=0.7, top_p=0.8, seed=0, max_new_tokens=120):
    torch.manual_seed(seed)
    prompt = render_prompt(user_text, thinking=thinking)
    batch = tokenizer(prompt, return_tensors="pt").to(device)
    kwargs = dict(max_new_tokens=max_new_tokens, do_sample=do_sample)
    if do_sample:
        kwargs.update(temperature=temperature, top_p=top_p, top_k=20)
    with torch.no_grad():
        output = model.generate(**batch, **kwargs)
    new = output[0, batch["input_ids"].shape[1]:]
    return tokenizer.decode(new, skip_special_tokens=True)

question = "Give exactly one legal five-letter Wordle guess. Output only the word."

print("GREEDY:")
print(generate(question, do_sample=False))

print("\nSAMPLED:")
for seed in range(3):
    print(seed, generate(question, do_sample=True, seed=seed))

GREEDY:
cute

SAMPLED:
0 answer
1 apple
2 apple


## 1.4 Thinking mode is an experimental variable

Qwen3 supports a thinking mode. We are **not** deciding yet whether it helps Wordle.

Run the same prompt with thinking enabled and disabled. Keep the distinction in mind because later training/evaluation must not accidentally change this variable.

In [11]:
prompt = "We are playing Wordle. Previous guess CRANE produced: C gray, R gray, A yellow, N green, E gray. Suggest the next guess and explain briefly."

print("NON-THINKING MODE")
print(generate(prompt, thinking=False, do_sample=True, seed=7))

print("\nTHINKING MODE")
print(generate(prompt, thinking=True, do_sample=True, temperature=0.6, top_p=0.95, seed=7, max_new_tokens=512))

NON-THINKING MODE
Based on the可以 you provided, here's a possible next guess for Wordle:

**Guess: CREESE**

### Explanation:
- **C** is correct (it matches the first letter of the word).
- **R** and **E** are also correct, but **E** is repeated.
- **G** and **N** are yellow and green, respectively.
- **S** is gray.

This guess is likely correct, and the next step would be to check the remaining letters. Let me know if you'd like a more refined guess!

THINKING MODE
<think>
Okay可以帮我分析一下Wordle的猜测情况。首先，用户之前的猜测是CRANE，现在给出的结果是：C gray, R gray, A yellow, N green, E gray。需要根据这些结果来推测下一步的猜测。

首先，我需要确定每个字母在单词中的位置。CRANE是单词，所以每个字母的位置是：C在第一个位置，R在第二个，A在第三个，N在第四个，E在第五个。现在给出的结果是每个字母的颜色，分别是灰、灰、黄、绿、灰。这里可能需要注意每个字母的颜色是否正确。

首先，C在单词中的第一个位置，结果是灰。根据Wordle的规则，如果猜到一个字母是正确的，并且位置正确，那么颜色应该和原单词相同。但这里C在第一个位置，结果是灰，所以可能原单词中的C是正确的位置，颜色是灰，所以C是正确的，颜色正确。所以第一个字母是C，颜色灰，正确。

接下来R在第二个位置，结果也是灰，所以R也是正确的，颜色灰。所以第二个字母R是正确的，颜色灰。

A在第三个位置，结果是黄，而原单词中的A在第三个位置，所以这里可能有问题。因为原单词中的A应该和结果中的颜色一致。但结果中显示A是黄色，而原单词中A是绿色，这说明可能颜色和字母不匹配？或者可能是其他情况？需

## 1.5 Baseline failure probes

We are not building the full benchmark yet. We are looking for obvious failure modes.

Run these prompts and classify each response:

- valid five-letter word?
- respects green positions?
- respects yellow letters?
- avoids known absent letters?
- obeys requested output format?

In [12]:
probes = [
    "Give exactly one legal five-letter Wordle guess. Output only the word.",
    "Wordle state: _ _ A _ _. Letter R is present but not position 2. C,E,S,T are absent. Give one next guess only.",
    "Wordle state: first letter is B, last letter is Y. A,E,I,O are absent. Give one next guess only.",
    "Wordle: guess EERIE gives feedback gray, yellow, gray, gray, green. Give one next guess only.",
]

for i, p in enumerate(probes, 1):
    print(f"\n--- Probe {i} ---")
    print(generate(p, thinking=False, do_sample=False))


--- Probe 1 ---
cute

--- Probe 2 ---
Based on the given information:

- The word is **A _ _ A _ _** (state: _ _ A _ _ _).
- Letter **R** is present but **not in position 2**.
- Letters **C, E, S, T** are **absent**.

So, the word must be:

**A _ _ A _ _**

We know that **R is not in position 2**, so it must be in position 1, 3, 4, or 5.

Also, **C, E, S, T are absent**,

--- Probe 3 ---
One possible next guess, given the constraints, is: **BAY**.

--- Probe 4 ---
Based on the feedback:  
- **EERIE** gives **gray**, **yellow**, **gray**, **gray**, and **green**.  

A possible next guess could be **EERIE** again, but since the user asked for **one next guess**, here's a valid option:  

**EERIE**.


## Lab 01 checkpoint

Write a short experiment note:

- What does Qwen's tokenizer do to five-letter words?
- Which failure mode seems most common?
- Does explicit character formatting appear worth testing later?
- What changes when thinking mode is enabled?
- What would you need to automate before calling any of this a benchmark?

The next lab finally changes weights.